# Tutorial 13 · Graph Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_13_Graph_Neural_Networks/tutorial_13_pytorch.ipynb)

How can a neural network use relationships rather than only independent feature vectors? Build message-passing layers, classify papers and entire graphs, and predict a molecular property.

**Start:** Open in Colab → connect → **Runtime → Run all**. Setup installs missing packages in Colab. No local Python installation, API key, Drive mount, or earlier notebook state is needed. While this repository is private, authorize GitHub access in Colab or download the notebook and use **File → Upload notebook**.

**Prerequisites:** PyTorch tensors, supervised learning, matrix multiplication, and train/validation/test splits.

**Resources:** CPU-compatible quick mode; expect several minutes plus package/data downloads. Cora, IMDB-BINARY, and a roughly 30 MB QM9 CSV download automatically. RDKit and PyTorch Geometric are installed if missing. GPU and larger mode are separate experiments.

**Learning goals:**
1. Read `x`, `edge_index`, and `batch`; trace a message from source to destination.
2. Implement GCN normalization and mean GraphSAGE; check against independent references.
3. Compare an MLP, GCN, and GraphSAGE on Cora's public node split.
4. Train GIN-0 and learned-epsilon GIN on IMDB-BINARY graphs.
5. Regress QM9 dipole moment in Debye, with training-only normalization and baselines.

The upstream lesson's missing implementations are completed here. Its repeated cross-validation/large sweeps become one seeded teaching split. QM9 uses SMILES from the DeepChem CSV instead of processing the full 3D archive; the section retains real QM9 targets but is **not** a 3D molecular benchmark.


## 1. Setup and reproducibility

Modern [PyG supports a minimal installation](https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html) without the optional compiled scatter/sparse extensions for these operations. Keep Colab's installed PyTorch; do not replace it with the upstream notebook's obsolete CUDA build. Start a fresh runtime if other experiments have changed dependencies.


In [ ]:
import importlib.util, subprocess, sys
packages={"torch":"torch", "torch_geometric":"torch-geometric>=2.6,<3", "rdkit":"rdkit",
          "numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib",
          "networkx":"networkx", "sklearn":"scikit-learn"}
missing=[package for module,package in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])


In [ ]:
import copy, hashlib, json, platform, random, time, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import torch
from torch import nn
import torch.nn.functional as F
import torch_geometric as pyg
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, GCNConv, SAGEConv, GINConv, global_add_pool, global_mean_pool
from torch_geometric.utils import add_self_loops, degree, to_undirected, to_dense_adj
from torch_geometric.datasets import Planetoid, TUDataset
from torch_geometric.transforms import NormalizeFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import Ridge
from rdkit import Chem, rdBase
from rdkit.Chem import Draw
from IPython.display import display
%matplotlib inline
SEED=42
QUICK_RUN=True
DEVICE_PREFERENCE="auto"
assert DEVICE_PREFERENCE in {"auto","cpu"}
device=torch.device("cuda" if DEVICE_PREFERENCE=="auto" and torch.cuda.is_available() else "cpu")
def seed_all(seed=SEED):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all();torch.set_num_threads(min(2,torch.get_num_threads()))
DATA_DIR=Path("data/tutorial_13");DATA_DIR.mkdir(parents=True,exist_ok=True)
OUT=Path("outputs/tutorial_13");OUT.mkdir(parents=True,exist_ok=True)
versions={"python":platform.python_version(),"torch":str(torch.__version__),"pyg":pyg.__version__,
          "rdkit":rdBase.rdkitVersion,"numpy":np.__version__,"pandas":pd.__version__,
          "matplotlib":matplotlib.__version__,"device":str(device),"quick_run":QUICK_RUN,"seed":SEED}
display(pd.Series(versions,name="Environment"));started=time.perf_counter();checks={};figures=[]
def finish(fig,name):
    fig.tight_layout();fig.savefig(OUT/name,bbox_inches="tight");figures.append(name);plt.show()
def cpu_state(model):return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}


## 2. See the graph, messages, and mini-batch

`x` has shape **[number of nodes, features]**. Each column of `edge_index` is **[source, destination]**; an undirected edge appears in both directions. Node IDs start at zero. The original tiny example referenced node 3 with only three feature rows; here every edge has a valid endpoint.

The first aggregation has no learned weights: each node receives the **sum of its neighbors' features**. Node 4 is isolated and receives zero. A mini-batch is a disjoint union: node IDs are shifted, and `batch` identifies each node's graph. No message may cross a graph boundary.


In [ ]:
edges=to_undirected(torch.tensor([[0,1,1,2],[1,2,3,3]],dtype=torch.long))
x=torch.tensor([[1.],[2.],[3.],[4.],[5.]])
toy=Data(x=x,edge_index=edges);assert toy.validate(raise_on_error=True)
messages=torch.zeros_like(x);messages.index_add_(0,edges[1],x[edges[0]])
assert torch.equal(messages[:,0],torch.tensor([2.,8.,6.,5.,0.]))
second=Data(x=torch.tensor([[6.],[7.],[8.]]),edge_index=to_undirected(torch.tensor([[0,1],[1,2]])))
mini=Batch.from_data_list([toy,second]);assert torch.equal(mini.batch[mini.edge_index[0]],mini.batch[mini.edge_index[1]])
fig,axes=plt.subplots(1,3,figsize=(15,4))
g=nx.Graph();g.add_nodes_from(range(5));g.add_edges_from(edges.T.tolist());pos=nx.spring_layout(g,seed=SEED)
for ax,values,title in zip(axes[:2],[x[:,0],messages[:,0]],["Before: node features","After: neighbor sum (no self-loop)"]):
    nx.draw_networkx(g,pos,ax=ax,labels={i:f"{i}: {v:g}" for i,v in enumerate(values.tolist())},node_color=values.numpy(),cmap="viridis",node_size=850,font_color="white",vmin=0,vmax=8)
    ax.set_title(title);ax.axis("off")
axes[2].imshow(to_dense_adj(mini.edge_index,max_num_nodes=8)[0],cmap="Blues");axes[2].axhline(4.5,color="red");axes[2].axvline(4.5,color="red")
axes[2].set(title="Batched adjacency: two disconnected blocks",xlabel="Source node",ylabel="Destination node")
finish(fig,"01_messages_and_batch.png")
display(pd.DataFrame({"node":range(8),"feature":mini.x[:,0],"graph_id":mini.batch}))
checks["manual_messages_and_batch_isolation"]=True


## 3. Implement GCN and GraphSAGE

GCN applies $H'=\hat D^{-1/2}(A+I)\hat D^{-1/2}HW+b$. Self-loops preserve a node's own feature; symmetric normalization limits the scale of high-degree neighborhoods. The implementation below assumes an undirected graph without existing self-loops, as in our toy graph and Cora.

Mean GraphSAGE applies $h'_i=W_{self}h_i+W_{neighbor}\operatorname{mean}_{j\in N(i)}h_j+b$. The root path remains separate. An isolated node has a zero neighbor average. ReLU and dropout belong to the enclosing model.

Tests compare GCN to dense matrix multiplication, GraphSAGE to PyG with copied weights, and verify that renumbering nodes renumbers outputs (permutation equivariance).


In [ ]:
class TeachingGCN(MessagePassing):
    def __init__(self,in_features,out_features):
        super().__init__(aggr="add")
        self.lin=nn.Linear(in_features,out_features,bias=False);self.bias=nn.Parameter(torch.zeros(out_features))
    def forward(self,x,edge_index):
        edge_index,_=add_self_loops(edge_index,num_nodes=x.size(0))
        source,target=edge_index
        inv=degree(target,x.size(0),dtype=x.dtype).clamp(min=1).pow(-0.5)
        return self.propagate(edge_index,x=self.lin(x),norm=inv[source]*inv[target])+self.bias
    def message(self,x_j,norm):return norm[:,None]*x_j
class TeachingSAGE(MessagePassing):
    def __init__(self,in_features,out_features):
        super().__init__(aggr="mean")
        self.neighbor=nn.Linear(in_features,out_features);self.root=nn.Linear(in_features,out_features,bias=False)
    def forward(self,x,edge_index):return self.neighbor(self.propagate(edge_index,x=x))+self.root(x)
    def message(self,x_j):return x_j
seed_all();gcn_check=TeachingGCN(1,3);sage_check=TeachingSAGE(1,3)
adj=to_dense_adj(edges,max_num_nodes=5)[0]+torch.eye(5);d=adj.sum(1).pow(-0.5)
dense=(d[:,None]*adj*d[None,:])@gcn_check.lin(x)+gcn_check.bias
assert torch.allclose(gcn_check(x,edges),dense,atol=1e-6)
ref=SAGEConv(1,3)
with torch.no_grad():
    ref.lin_l.weight.copy_(sage_check.neighbor.weight);ref.lin_l.bias.copy_(sage_check.neighbor.bias)
    ref.lin_r.weight.copy_(sage_check.root.weight)
assert torch.allclose(sage_check(x,edges),ref(x,edges),atol=1e-6)
perm=torch.tensor([3,0,4,1,2]);inverse=torch.argsort(perm)
for layer in [gcn_check,sage_check]:
    assert torch.allclose(layer(x[perm],inverse[edges]),layer(x,edges)[perm],atol=1e-6)
    layer(x,edges).square().mean().backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in layer.parameters())
checks["gcn_dense_sage_reference_gradients_equivariance"]=True
print("PASS: dense GCN, reference GraphSAGE, gradients, and node-permutation checks")


## 4. Cora: classify nodes in one shared graph

Cora contains papers (nodes), citation links (edges), bag-of-words features, and topic labels. This is **transductive** learning: all node features and edges are visible during training, but only training labels enter the loss. GraphSAGE's architecture can support inductive use; this experiment does not test unseen graphs.

Use the fixed public masks: training labels for gradients, validation loss for checkpoint selection, and test labels only after selection. Row-wise feature normalization uses each node's own feature sum. Compare against a feature-only MLP and the training-label majority. The three models use the same split; this single seed is not a published benchmark or a broad architecture ranking.


In [ ]:
cora_dataset=Planetoid(str(DATA_DIR/"Cora"),"Cora",split="public",transform=NormalizeFeatures())
cora=cora_dataset[0];assert cora.validate(raise_on_error=True)
assert not (cora.edge_index[0]==cora.edge_index[1]).any()
masks=[cora.train_mask,cora.val_mask,cora.test_mask]
assert not any((masks[i]&masks[j]).any() for i in range(3) for j in range(i))
print(cora);print("Train / validation / test:",[int(m.sum()) for m in masks])
class NodeClassifier(nn.Module):
    def __init__(self,kind):
        super().__init__();self.kind=kind
        cls={"MLP":nn.Linear,"GCN":TeachingGCN,"GraphSAGE":TeachingSAGE}[kind]
        self.first=cls(cora.num_features,32);self.second=cls(32,cora_dataset.num_classes)
    def forward(self,data):
        if self.kind=="MLP":h=self.first(data.x)
        else:h=self.first(data.x,data.edge_index)
        h=F.dropout(F.relu(h),p=0.5,training=self.training)
        return self.second(h) if self.kind=="MLP" else self.second(h,data.edge_index)
node_models={};node_histories={};node_rows=[];cora_gpu=cora.to(device)
for kind in ["MLP","GCN","GraphSAGE"]:
    seed_all();model=NodeClassifier(kind).to(device);opt=torch.optim.Adam(model.parameters(),lr=0.01,weight_decay=5e-4)
    history=[];best=float("inf");best_epoch=0
    for epoch in range(1,(120 if QUICK_RUN else 300)+1):
        model.train();opt.zero_grad();logits=model(cora_gpu)
        loss=F.cross_entropy(logits[cora_gpu.train_mask],cora_gpu.y[cora_gpu.train_mask]);loss.backward();opt.step()
        model.eval()
        with torch.no_grad():vl=float(F.cross_entropy(model(cora_gpu)[cora_gpu.val_mask],cora_gpu.y[cora_gpu.val_mask]))
        assert np.isfinite(vl);history.append([float(loss.detach()),vl])
        if vl<best:best=vl;state=cpu_state(model);best_epoch=epoch
    model.load_state_dict(state);model.eval()
    with torch.no_grad():prediction=model(cora_gpu).argmax(1)
    row={"model":kind,"selected_epoch":best_epoch,"val_loss":best}
    for name,mask in zip(["train","val","test"],masks):row[name+"_accuracy"]=float((prediction[mask]==cora_gpu.y[mask]).float().mean())
    node_rows.append(row);node_models[kind]=model.cpu();node_histories[kind]=np.array(history)
    print(kind,row)
cora=cora_gpu.cpu();majority=int(cora.y[cora.train_mask].bincount().argmax())
node_baseline=float((cora.y[cora.test_mask]==majority).float().mean())
checks["cora_masks_disjoint"]=True


### Inspect errors and learning curves

The confusion matrix is for the architecture selected by **validation loss**, not by test accuracy. If a graph model improves on the MLP, that is evidence that citation structure helped on this split; it does not establish that edges always help.

**Baseline detail:** Cora has 20 training labels per class. The majority rule therefore has a seven-way tie; `argmax` selects class ID 0. Its 13% test accuracy is a tie-breaking artifact, so the feature-only MLP is the more informative comparator.


In [ ]:
node_table=pd.DataFrame(node_rows);display(node_table)
selected_node=min(node_rows,key=lambda r:r["val_loss"])["model"]
with torch.no_grad():node_pred=node_models[selected_node](cora).argmax(1)
fig,axes=plt.subplots(1,3,figsize=(16,4))
for kind,h in node_histories.items():axes[0].plot(np.arange(1,len(h)+1),h[:,1],label=kind)
axes[0].set(title="Cora validation loss",xlabel="Epoch",ylabel="Cross entropy");axes[0].legend()
axes[1].bar(node_table.model,node_table.test_accuracy);axes[1].axhline(node_baseline,color="red",ls="--",label="Training majority")
axes[1].set(title="Held-out node accuracy",ylim=(0,1),ylabel="Accuracy");axes[1].legend()
cm=confusion_matrix(cora.y[cora.test_mask],node_pred[cora.test_mask],labels=list(range(cora_dataset.num_classes)))
axes[2].imshow(cm,cmap="Blues");axes[2].set(title=f"Validation-selected {selected_node}: test counts",xlabel="Predicted topic ID",ylabel="True topic ID")
for (i,j),v in np.ndenumerate(cm):axes[2].text(j,i,str(v),ha="center",va="center",color="white" if v>cm.max()/2 else "black",fontsize=8)
finish(fig,"02_cora_comparison.png");node_table.to_csv(OUT/"cora_metrics.csv",index=False)


## 5. IMDB-BINARY: classify whole graphs with GIN

A movie collaboration graph has actors as nodes and collaboration edges; labels represent the dataset's two movie categories. There are no supplied node attributes. We use a constant and log-degree, then standardize **log-degree using training graphs only**.

GIN updates $h'_i=\operatorname{MLP}((1+\epsilon)h_i+\sum_{j\in N(i)}h_j)$. Compare fixed $\epsilon=0$ with learned epsilon. Sum pooling gives one vector per graph. Sum and mean are not interchangeable: two identical node vectors have the same mean as one, but twice its sum. GIN's expressivity has assumptions and does not imply distinguishing every pair of non-isomorphic graphs.

Use a stratified 70/15/15 split and a validation-selected epoch. Graph indices are disjoint; the original IMDB dataset may contain isomorphic graphs across splits. We do **not** claim an isomorphism-clean benchmark. A training-majority baseline is included.


In [ ]:
imdb_raw=TUDataset(str(DATA_DIR/"IMDB"),name="IMDB-BINARY")
labels=np.array([int(g.y.item()) for g in imdb_raw]);indices=np.arange(len(labels))
train_ids,hold_ids=train_test_split(indices,test_size=0.3,random_state=SEED,stratify=labels)
val_ids,test_ids=train_test_split(hold_ids,test_size=0.5,random_state=SEED,stratify=labels[hold_ids])
assert not (set(train_ids)&set(val_ids) or set(train_ids)&set(test_ids) or set(val_ids)&set(test_ids))
train_degrees=torch.cat([torch.log1p(degree(imdb_raw[int(i)].edge_index[0],imdb_raw[int(i)].num_nodes)) for i in train_ids])
deg_mean=train_degrees.mean();deg_std=train_degrees.std().clamp(min=1e-6)
def imdb_graph(i):
    g=imdb_raw[int(i)].clone();v=torch.log1p(degree(g.edge_index[0],g.num_nodes))
    g.x=torch.stack([torch.ones_like(v),(v-deg_mean)/deg_std],1);return g
imdb_sets=[[imdb_graph(i) for i in ids] for ids in [train_ids,val_ids,test_ids]]
print("IMDB split sizes:",[len(s) for s in imdb_sets])
class GraphGIN(nn.Module):
    def __init__(self,train_eps):
        super().__init__();self.convs=nn.ModuleList()
        for fin in [2,32,32]:self.convs.append(GINConv(nn.Sequential(nn.Linear(fin,32),nn.ReLU(),nn.Linear(32,32)),train_eps=train_eps))
        self.head=nn.Sequential(nn.Linear(32,32),nn.ReLU(),nn.Dropout(0.3),nn.Linear(32,2))
    def forward(self,g):
        x=g.x
        for conv in self.convs:x=F.relu(conv(x,g.edge_index))
        return self.head(global_add_pool(x,g.batch))
def graph_evaluate(model,loader,regression=False):
    model.eval();ys=[];preds=[]
    with torch.no_grad():
        for batch in loader:
            batch=batch.to(device);ys.append(batch.y.view(-1).cpu());preds.append(model(batch).cpu())
    return torch.cat(ys),torch.cat(preds)
gin_models={};gin_rows=[];gin_histories={}
for train_eps in [False,True]:
    seed_all();name="GIN-eps" if train_eps else "GIN-0";model=GraphGIN(train_eps).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=0.003);best=float("inf");history=[]
    loader=DataLoader(imdb_sets[0],batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED))
    for epoch in range(1,(35 if QUICK_RUN else 120)+1):
        model.train();total=0
        for batch in loader:
            batch=batch.to(device);opt.zero_grad();loss=F.cross_entropy(model(batch),batch.y);loss.backward();opt.step();total+=float(loss.detach())*batch.num_graphs
        y,p=graph_evaluate(model,DataLoader(imdb_sets[1],batch_size=128));vl=float(F.cross_entropy(p,y));assert np.isfinite(vl)
        history.append([total/len(imdb_sets[0]),vl])
        if vl<best:best=vl;state=cpu_state(model);best_epoch=epoch
    model.load_state_dict(state);y,p=graph_evaluate(model,DataLoader(imdb_sets[2],batch_size=128))
    gin_rows.append({"model":name,"selected_epoch":best_epoch,"val_loss":best,"test_accuracy":float((p.argmax(1)==y).float().mean())})
    gin_models[name]=model.cpu();gin_histories[name]=np.array(history)
gin_table=pd.DataFrame(gin_rows);display(gin_table)
imdb_majority=int(np.bincount(labels[train_ids]).argmax());imdb_baseline=float(np.mean(labels[test_ids]==imdb_majority))
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,h in gin_histories.items():axes[0].plot(np.arange(1,len(h)+1),h[:,1],label=name)
axes[0].set(title="IMDB validation loss",xlabel="Epoch",ylabel="Cross entropy");axes[0].legend()
axes[1].bar(gin_table.model,gin_table.test_accuracy);axes[1].axhline(imdb_baseline,color="red",ls="--",label="Training majority")
axes[1].set(title="Held-out graph accuracy",ylim=(0,1),ylabel="Accuracy");axes[1].legend();finish(fig,"03_gin_comparison.png")
gin_table.to_csv(OUT/"imdb_metrics.csv",index=False)
probe=Batch.from_data_list(imdb_sets[2][:3]);model=gin_models["GIN-eps"].eval()
with torch.no_grad():
    separate=torch.cat([model(Batch.from_data_list([g])) for g in imdb_sets[2][:3]])
    assert torch.allclose(model(probe),separate,atol=2e-5)
    gp=imdb_sets[2][0].clone();p=torch.randperm(gp.num_nodes);inv=torch.argsort(p);gp.x=gp.x[p];gp.edge_index=inv[gp.edge_index]
    assert torch.allclose(model(Batch.from_data_list([gp])),separate[:1],atol=2e-5)
checks["graph_split_batch_equivalence_pooling_invariance"]=True


## 6. QM9: molecular dipole regression

Keep real QM9 data while avoiding the large 3D preprocessing step: download the [DeepChem QM9 CSV](https://github.com/deepchem/deepchem/blob/master/deepchem/molnet/load_function/qm9_datasets.py), select a seeded subset, parse SMILES with RDKit, and predict **dipole moment `mu`, in Debye (D)**. The CSV includes other calculated properties; none enters the features.

Each atom has H/C/N/O/F one-hot features, total hydrogen count, aromaticity, and scaled formal charge. Bonds become undirected edges. This deliberately simple GCN ignores bond types and 3D coordinates, both relevant to dipole moments. It is a molecular-property teaching example, not a crystal/materials discovery model.

Select 3,000 unique canonical SMILES in quick mode (10,000 otherwise), then split 70/15/15. Invalid SMILES are skipped and counted. Canonical SMILES do not cross splits. Random molecule splits can share scaffolds, so they do not measure extrapolation to new chemistry. Target mean/std come from training molecules only. Compare the GCN to the training mean and a Ridge model using atom-count features; choose Ridge's regularization on validation MAE.


In [ ]:
qm9_path=DATA_DIR/"qm9.csv"
if not qm9_path.exists():
    temporary=qm9_path.with_suffix(".part");urllib.request.urlretrieve("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv",temporary);temporary.replace(qm9_path)
qm9_hash=hashlib.sha256(qm9_path.read_bytes()).hexdigest()
frame=pd.read_csv(qm9_path);assert {"smiles","mu"}<=set(frame.columns)
assert np.isfinite(frame["mu"]).all()
order=np.random.default_rng(SEED).permutation(len(frame));molecules=[];canonical=[];seen=set();invalid=0;duplicates=0;atom_counts=[];mol_objects=[]
atom_types=[1,6,7,8,9]
for row_id in order:
    row=frame.iloc[row_id];mol=Chem.MolFromSmiles(row.smiles)
    if mol is None:invalid+=1;continue
    key=Chem.MolToSmiles(mol,isomericSmiles=True)
    if key in seen:duplicates+=1;continue
    seen.add(key);features=[]
    for atom in mol.GetAtoms():
        z=atom.GetAtomicNum();assert z in atom_types
        features.append([float(z==t) for t in atom_types]+[atom.GetTotalNumHs()/4,float(atom.GetIsAromatic()),atom.GetFormalCharge()/2])
    bond_edges=[]
    for bond in mol.GetBonds():
        a,b=bond.GetBeginAtomIdx(),bond.GetEndAtomIdx();bond_edges.extend([(a,b),(b,a)])
    ei=torch.tensor(bond_edges,dtype=torch.long).reshape(-1,2).T.contiguous()
    graph=Data(x=torch.tensor(features,dtype=torch.float32),edge_index=ei,y=torch.tensor([float(row.mu)],dtype=torch.float32))
    assert graph.validate(raise_on_error=True)
    molecules.append(graph);canonical.append(key);mol_objects.append(mol)
    atom_counts.append([sum(a.GetAtomicNum()==z for a in mol.GetAtoms()) for z in atom_types]+[sum(a.GetTotalNumHs() for a in mol.GetAtoms())])
    if len(molecules)==(3000 if QUICK_RUN else 10000):break
assert len(molecules)==(3000 if QUICK_RUN else 10000)
qtrain,qhold=train_test_split(np.arange(len(molecules)),test_size=0.3,random_state=SEED)
qval,qtest=train_test_split(qhold,test_size=0.5,random_state=SEED)
qsets=[[molecules[int(i)] for i in ids] for ids in [qtrain,qval,qtest]]
keys=[set(canonical[int(i)] for i in ids) for ids in [qtrain,qval,qtest]]
assert not (keys[0]&keys[1] or keys[0]&keys[2] or keys[1]&keys[2])
y_all=np.array([g.y.item() for g in molecules]);target_mean=float(y_all[qtrain].mean());target_std=float(y_all[qtrain].std())
assert target_std>0
print("QM9 split sizes:",[len(s) for s in qsets],"invalid skipped:",invalid,"duplicates skipped:",duplicates)
print("Training target mean/std (D):",target_mean,target_std,"CSV SHA256:",qm9_hash)
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,idx in zip(axes,qtrain[:3]):
    ax.imshow(Draw.MolToImage(mol_objects[int(idx)],size=(350,280)));ax.axis("off");ax.set_title(f"Training molecule {idx}: mu = {y_all[idx]:.2f} D")
finish(fig,"04_qm9_molecules.png")
fig,ax=plt.subplots(figsize=(7,4))
for name,ids in zip(["Train","Validation","Test"],[qtrain,qval,qtest]):ax.hist(y_all[ids],bins=np.linspace(0,max(y_all),25),density=True,histtype="step",label=name)
ax.set(xlabel="Dipole moment (D)",ylabel="Density",title="QM9 target distributions");ax.legend();finish(fig,"05_qm9_targets.png")
checks["qm9_unique_smiles_disjoint_training_only_scaling"]=True


### Train, select, and evaluate in physical units

Two graph convolutions produce node embeddings; **mean pooling** forms a graph vector. The head predicts standardized dipole moment with MSE loss. Convert predictions back to Debye before reporting MAE/RMSE. Validation MAE selects an epoch; the held-out test set is used only afterward. Mean pooling and our restricted features may underfit molecular size and geometry. A lower training loss alone does not establish useful prediction.


In [ ]:
class MolecularGCN(nn.Module):
    def __init__(self):
        super().__init__();self.conv1=GCNConv(8,64);self.conv2=GCNConv(64,64)
        self.head=nn.Sequential(nn.Linear(64,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self,g):
        x=F.relu(self.conv1(g.x,g.edge_index));x=F.relu(self.conv2(x,g.edge_index))
        return self.head(global_mean_pool(x,g.batch)).view(-1)
seed_all();molecular_model=MolecularGCN().to(device);optimizer=torch.optim.Adam(molecular_model.parameters(),lr=0.003)
loader=DataLoader(qsets[0],batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED))
qhistory=[];best=float("inf")
for epoch in range(1,(45 if QUICK_RUN else 150)+1):
    molecular_model.train();total=0
    for batch in loader:
        batch=batch.to(device);optimizer.zero_grad();target=(batch.y.view(-1)-target_mean)/target_std
        loss=F.mse_loss(molecular_model(batch),target);loss.backward();optimizer.step();total+=float(loss.detach())*batch.num_graphs
    y,p=graph_evaluate(molecular_model,DataLoader(qsets[1],batch_size=256));mae=float((p*target_std+target_mean-y).abs().mean())
    assert np.isfinite(mae);qhistory.append([total/len(qsets[0]),mae])
    if mae<best:best=mae;qstate=cpu_state(molecular_model);qepoch=epoch
molecular_model.load_state_dict(qstate);qt,qp=graph_evaluate(molecular_model,DataLoader(qsets[2],batch_size=256));qp=qp.numpy()*target_std+target_mean;qt=qt.numpy()
molecular_model=molecular_model.cpu();counts=np.array(atom_counts,dtype=float)
ridge_candidates=[]
for alpha in [0.1,1,10,100]:
    reg=Ridge(alpha=alpha).fit(counts[qtrain],y_all[qtrain]);val_mae=float(np.abs(reg.predict(counts[qval])-y_all[qval]).mean());ridge_candidates.append((val_mae,alpha,reg))
ridge_val,ridge_alpha,ridge=min(ridge_candidates,key=lambda item:item[0]);ridge_pred=ridge.predict(counts[qtest])
qrows=[]
for name,pred in [("Training mean",np.full_like(qt,target_mean)),("Atom-count Ridge",ridge_pred),("Molecular GCN",qp)]:
    qrows.append({"model":name,"test_mae_D":float(np.abs(pred-qt).mean()),"test_rmse_D":float(np.sqrt(np.mean((pred-qt)**2)))})
qtable=pd.DataFrame(qrows);display(qtable);print("Selected GCN epoch:",qepoch,"Ridge alpha:",ridge_alpha)
fig,axes=plt.subplots(1,3,figsize=(16,4));qh=np.array(qhistory)
axes[0].plot(np.arange(1,len(qh)+1),qh[:,1]);axes[0].axvline(qepoch,color="red",ls="--")
axes[0].set(title="QM9 validation MAE",xlabel="Epoch",ylabel="MAE (D)")
axes[1].scatter(qt,qp,s=12,alpha=0.5);lo=min(qt.min(),qp.min());hi=max(qt.max(),qp.max());axes[1].plot([lo,hi],[lo,hi],"k--")
axes[1].set(title="Held-out molecular predictions",xlabel="True dipole (D)",ylabel="Predicted dipole (D)")
axes[2].barh(qtable.model,qtable.test_mae_D);axes[2].set(title="Test error: lower is better",xlabel="MAE (D)")
finish(fig,"06_qm9_regression.png")
qtable.to_csv(OUT/"qm9_metrics.csv",index=False)
pd.DataFrame({"smiles":[canonical[int(i)] for i in qtest],"true_mu_D":qt,"gcn_mu_D":qp,"ridge_mu_D":ridge_pred}).to_csv(OUT/"qm9_predictions.csv",index=False)


## 7. Save, reload, and audit

Each checkpoint stores only a state dictionary and basic metadata. The regression checkpoint includes its target scaling; the GIN checkpoint includes degree scaling. A model file alone does not replace its architecture or preprocessing. Reload on CPU and compare predictions on fixed probes, then save a machine-readable run summary. Download `outputs/tutorial_13/` before your runtime disconnects. Load only checkpoints you trust.


In [ ]:
reload_names=[]
def round_trip(name,model,factory,probe,metadata):
    model.cpu().eval();path=OUT/(name+".pt")
    torch.save({"state_dict":cpu_state(model),"metadata":metadata},path)
    saved=torch.load(path,map_location="cpu",weights_only=True);restored=factory().eval();restored.load_state_dict(saved["state_dict"])
    with torch.no_grad():assert torch.allclose(model(probe),restored(probe),atol=1e-6)
    assert saved["metadata"]==metadata;reload_names.append(name)
for name,model in node_models.items():round_trip("cora_"+name,model,lambda k=name:NodeClassifier(k),cora,{"features":"row-normalized Cora","kind":name})
for name,model in gin_models.items():round_trip("imdb_"+name,model,lambda n=name:GraphGIN(n=="GIN-eps"),Batch.from_data_list(imdb_sets[2][:4]),{"degree_mean":float(deg_mean),"degree_std":float(deg_std)})
round_trip("qm9_GCN",molecular_model,MolecularGCN,Batch.from_data_list(qsets[2][:4]),{"target":"mu","unit":"Debye","target_mean":target_mean,"target_std":target_std,"features":"H C N O F one-hot; total H/4; aromatic; formal charge/2"})
checks["six_checkpoint_cpu_reload"]=len(reload_names)==6
assert all(checks.values())
summary={"environment":versions,"checks":checks,"reloads":reload_names,"figures":figures,
         "cora":{"results":node_rows,"test_majority_accuracy":node_baseline,"selected_model":selected_node},
         "imdb":{"results":gin_rows,"test_majority_accuracy":imdb_baseline,"split_sizes":[len(s) for s in imdb_sets]},
         "qm9":{"results":qrows,"split_sizes":[len(s) for s in qsets],"csv_sha256":qm9_hash,"invalid_smiles_skipped":invalid,"duplicates_skipped":duplicates,"selected_epoch":qepoch,"ridge_alpha":ridge_alpha,"validation_mae_D":best},
         "elapsed_seconds_after_imports":time.perf_counter()-started}
(OUT/"run_summary.json").write_text(json.dumps(summary,indent=2))
print("TUTORIAL_13_COMPLETE");print(json.dumps(summary,indent=2));print("All checks passed; outputs saved to",OUT)


## Observed quick-mode Colab CPU run (2026-10-13)

All 10 code cells and six CPU checkpoint reloads passed; six generated figures were inspected. In this single seeded run:

| Task | Model | Held-out result |
|---|---|---|
| Cora | MLP / GCN / GraphSAGE | 58.8% / 81.2% / 80.8% accuracy |
| IMDB-BINARY | GIN-0 / GIN-eps | 69.3% / 70.7% accuracy (majority: 50%) |
| QM9 dipole | GCN / atom-count Ridge / training mean | 0.763 / 1.038 / 1.196 D MAE |

GraphSAGE was selected by Cora validation loss, although GCN happened to score slightly higher on the held-out test set; the test result does not change selection. GIN has large initial loss spikes with sum pooling before settling. QM9 predictions underestimate rare large dipoles: the lower average error does not establish accuracy in that tail. The run took about 46 seconds after imports on this CPU; setup/download speeds and other runtimes can differ. GPU and larger mode were not tested.

## Interpretation and exercises

- Which Cora model improves over feature-only prediction? Why is this still a transductive experiment?
- Replace graph sum pooling with mean pooling. Explain the information lost before testing it.
- Inspect where QM9 predictions regress toward the mean. Which missing inputs could explain high-dipole errors?
- Compare five seeds and report uncertainty. Do not choose seeds or hyperparameters using test scores.
- As a separate experiment, create scaffold-disjoint QM9 splits, encode bond types, and incorporate 3D geometry. These are not implemented or verified by this default run.

**Limits:** Small subsets, one seed, simplified models, possible IMDB isomorphism overlap, and random molecular splits prevent benchmark or materials-discovery claims. An execution pass verifies a workflow, not scientific adequacy. GPU and `QUICK_RUN=False` require separate validation.

## Troubleshooting and attribution

Run from the top after a restart. If a download fails, check connectivity and rerun; do not silently replace real data with synthetic data. If imports fail after other package experiments, start a fresh runtime. Download output files before Colab disconnects.

Adapted from [PEESE Tutorial 13](https://github.com/PEESEgroup/SysEn6888/tree/0b326ded95e415d7a60660879aaa737e3c7d79ea/Tutorial_13_Graph_Neural_Networks), Copyright (c) 2025 PEESE; its MIT notice is retained in `LICENSE`. This independent adaptation does not imply endorsement.

References: [GCN](https://arxiv.org/abs/1609.02907), [GraphSAGE](https://arxiv.org/abs/1706.02216), [GIN](https://arxiv.org/abs/1810.00826), [PyG message passing](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_gnn.html), [Planetoid/Cora](https://github.com/kimiyoung/planetoid), [TU datasets](https://chrsmrrs.github.io/datasets/), [QM9 original data](https://doi.org/10.1038/sdata.2014.22), [DeepChem QM9 loader and units](https://github.com/deepchem/deepchem/blob/master/deepchem/molnet/load_function/qm9_datasets.py). Dataset/software terms remain separate from this repository's code license. Data and checkpoints are not committed to GitHub.
